# Porównanie architektur transfer learning

## Cel eksperymentu

Celem eksperymentu jest porównanie skuteczności modeli ResNet50, EfficientNet-B0 oraz MobileNetV3 w klasyfikacji 92 klas znaków drogowych.

Wszystkie modele wykorzystują wstępnie wytrenowane wagi ImageNet oraz tę samą strategię kompensacji niezbalansowania klas — `WeightedRandomSampler`.

Eksperyment składa się z dwóch etapów:

1. trening głowy klasyfikacyjnej przy zamrożonym backbone,
2. fine-tuning najlepszego modelu poprzez odmrożenie końcowych warstw.

Wybór architektury odbywa się na podstawie wartości `macro F1` na zbiorze walidacyjnym.

Zbiór `test_final` jest wykorzystywany dopiero do końcowej oceny wybranej konfiguracji.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

from src.data.dataset import (
    TrafficSignDataset,
)

from src.data.transforms import (
    build_train_transform,
    build_eval_transform,
)

from src.data.sampling import (
    build_weighted_sampler,
)

from src.models.resnet50 import (
    create_resnet50,
    freeze_resnet50_backbone,
    unfreeze_resnet50_last_block,
)

from src.models.efficientnet_b0 import (
    create_efficientnet_b0,
    freeze_efficientnet_b0_backbone,
    unfreeze_efficientnet_b0_last_block,
)

from src.models.mobilenet_v3 import (
    create_mobilenet_v3,
    freeze_mobilenet_v3_backbone,
    unfreeze_mobilenet_v3_last_block,
)

from src.training.trainer import (
    fit,
)

from src.evaluation.metrics import (
    predict,
    evaluate_predictions,
)

from src.utils.seed import (
    set_seed,
)

## 1. Konfiguracja eksperymentu

### Faza 1

Zamrożony backbone, trenowana wyłącznie głowica klasyfikacyjna.

- epoki: 10
- learning rate: 0.001
- batch size: 32
- WeightedRandomSampler
- kryterium wyboru: validation macro F1

### Faza 2

Fine-tuning najlepszego modelu.

- odmrożenie końcowych bloków,
- epoki: 5
- learning rate: 0.00001,
- WeightedRandomSampler,
- kryterium wyboru: validation macro F1.

In [ ]:
set_seed(42)

METADATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "data"
    / "processed"
    / "metadata"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
)

METRICS_DIR = (
    RESULTS_DIR
    / "metrics"
)

HISTORIES_DIR = (
    RESULTS_DIR
    / "histories"
)

CONFUSION_DIR = (
    RESULTS_DIR
    / "confusion_matrices"
)

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "models"
    / "checkpoints"
)

for directory in [
    METRICS_DIR,
    HISTORIES_DIR,
    CONFUSION_DIR,
    CHECKPOINT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

TRAIN_MANIFEST = (
    METADATA_DIR
    / "07_train.csv"
)

VALIDATION_MANIFEST = (
    METADATA_DIR
    / "08_validation.csv"
)

TEST_MANIFEST = (
    METADATA_DIR
    / "09_test_final.csv"
)

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
FROZEN_EPOCHS = 10
FINE_TUNE_EPOCHS = 5
FROZEN_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-5
RANDOM_SEED = 42
PRETRAINED = True

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Urządzenie:", DEVICE)
print("Input:", IMAGE_SIZE)
print("Batch:", BATCH_SIZE)
print("Faza 1 — epoki:", FROZEN_EPOCHS)
print("Faza 2 — epoki:", FINE_TUNE_EPOCHS)
print(
    "Faza 1 — learning rate:",
    FROZEN_LEARNING_RATE,
)
print(
    "Faza 2 — learning rate:",
    FINE_TUNE_LEARNING_RATE,
)

Urządzenie: cpu
Input: 224
Batch: 32
Faza 1 — epoki: 10
Faza 2 — epoki: 5
Faza 1 — learning rate: 0.001
Faza 2 — learning rate: 1e-05


## 2. Mapowanie klas

In [3]:
train_df = pd.read_csv(
    TRAIN_MANIFEST
)

validation_df = pd.read_csv(
    VALIDATION_MANIFEST
)

test_df = pd.read_csv(
    TEST_MANIFEST
)

class_codes = sorted(
    set(
        train_df["class_code"].astype(str)
    )
    |
    set(
        validation_df["class_code"].astype(str)
    )
    |
    set(
        test_df["class_code"].astype(str)
    )
)

class_to_idx = {
    class_code: index
    for index, class_code
    in enumerate(class_codes)
}

idx_to_class = {
    index: class_code
    for class_code, index
    in class_to_idx.items()
}

print(
    "Liczba klas:",
    len(class_to_idx)
)

Liczba klas: 92


## 3. Przygotowanie Datasetów i DataLoaderów

In [4]:
train_dataset = TrafficSignDataset(
    TRAIN_MANIFEST,
    class_to_idx,
    transform=build_train_transform(
        IMAGE_SIZE
    ),
)

validation_dataset = TrafficSignDataset(
    VALIDATION_MANIFEST,
    class_to_idx,
    transform=build_eval_transform(
        IMAGE_SIZE
    ),
)

test_dataset = TrafficSignDataset(
    TEST_MANIFEST,
    class_to_idx,
    transform=build_eval_transform(
        IMAGE_SIZE
    ),
)

weighted_sampler = (
    build_weighted_sampler(
        TRAIN_MANIFEST
    )
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=weighted_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type == "cuda"
    ),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type == "cuda"
    ),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type == "cuda"
    ),
)

print(
    "TRAIN:",
    len(train_dataset)
)

print(
    "VALIDATION:",
    len(validation_dataset)
)

print(
    "TEST:",
    len(test_dataset)
)

TRAIN: 14232
VALIDATION: 2512
TEST: 4298


## 4. Faza 1 — zamrożony backbone

W pierwszym etapie wykorzystywane są reprezentacje wyuczone na zbiorze ImageNet. Parametry backbone'u pozostają zamrożone, natomiast uczeniu podlega warstwa klasyfikacyjna dostosowana do 92 klas znaków drogowych.

In [5]:
def run_frozen_experiment(
    model,
    freeze_function,
    model_name,
):

    set_seed(
        RANDOM_SEED
    )

    model = freeze_function(
        model
    )

    model = model.to(
        DEVICE
    )

    criterion = (
        nn.CrossEntropyLoss()
    )

    optimizer = torch.optim.Adam(
        filter(
            lambda parameter:
            parameter.requires_grad,
            model.parameters(),
        ),
        lr=FROZEN_LEARNING_RATE,
    )

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"{model_name}_frozen_best.pt"
    )

    history = fit(
        model=model,
        train_loader=train_loader,
        val_loader=validation_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        epochs=FROZEN_EPOCHS,
        checkpoint_path=checkpoint_path,
    )

    y_true, y_pred, inference_time = (
        predict(
            model,
            validation_loader,
            DEVICE,
        )
    )

    metrics = evaluate_predictions(
        y_true,
        y_pred,
    )

    result = {
        "model": model_name,
        "phase": "frozen",
        "accuracy": metrics[
            "accuracy"
        ],
        "macro_precision": metrics[
            "macro_precision"
        ],
        "macro_recall": metrics[
            "macro_recall"
        ],
        "macro_f1": metrics[
            "macro_f1"
        ],
        "best_val_macro_f1": history[
            "best_val_macro_f1"
        ],
        "training_time_seconds": history[
            "training_time_seconds"
        ],
    }

    (
        HISTORIES_DIR
        / f"{model_name}_frozen.json"
    ).write_text(
        json.dumps(
            history,
            indent=2,
        ),
        encoding="utf-8",
    )

    return (
        model,
        history,
        result,
    )

In [6]:
resnet50_model = create_resnet50(
    num_classes=len(
        class_to_idx
    ),
    pretrained=PRETRAINED,
)

(
    resnet50_model,
    resnet50_history,
    resnet50_result,
) = run_frozen_experiment(
    model=resnet50_model,
    freeze_function=(
        freeze_resnet50_backbone
    ),
    model_name="resnet50",
)

print(
    json.dumps(
        resnet50_result,
        indent=2,
    )
)

Postęp treningu:   0%|          | 0/10 [00:00<?, ?it/s]

Epoka 1/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 2/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 3/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 4/10:   0%|          | 0/445 [00:00<?, ?it/s]

...

Epoka 7/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 8/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 9/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 10/10:   0%|          | 0/445 [00:00<?, ?it/s]

{
  "model": "resnet50",
  "phase": "frozen",
  "accuracy": 0.8722133757961783,
  "macro_precision": 0.8661945238676635,
  "macro_recall": 0.869549052875761,
  "macro_f1": 0.8589602068951625,
  "best_val_macro_f1": 0.8589602068951625,
  "training_time_seconds": 16647.1410729
}


In [7]:
efficientnet_model = (
    create_efficientnet_b0(
        num_classes=len(
            class_to_idx
        ),
        pretrained=PRETRAINED,
    )
)

(
    efficientnet_model,
    efficientnet_history,
    efficientnet_result,
) = run_frozen_experiment(
    model=efficientnet_model,
    freeze_function=(
        freeze_efficientnet_b0_backbone
    ),
    model_name="efficientnet_b0",
)

print(
    json.dumps(
        efficientnet_result,
        indent=2,
    )
)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\Dell/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 32.4MB/s]


Postęp treningu:   0%|          | 0/10 [00:00<?, ?it/s]

Epoka 1/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 2/10:   0%|          | 0/445 [00:00<?, ?it/s]

...

Epoka 7/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 8/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 9/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 10/10:   0%|          | 0/445 [00:00<?, ?it/s]

{
  "model": "efficientnet_b0",
  "phase": "frozen",
  "accuracy": 0.8471337579617835,
  "macro_precision": 0.8034885842357326,
  "macro_recall": 0.8488395139115916,
  "macro_f1": 0.8193001792406567,
  "best_val_macro_f1": 0.8193001792406567,
  "training_time_seconds": 7143.604567399998
}


In [8]:
mobilenet_model = (
    create_mobilenet_v3(
        num_classes=len(
            class_to_idx
        ),
        pretrained=PRETRAINED,
    )
)

(
    mobilenet_model,
    mobilenet_history,
    mobilenet_result,
) = run_frozen_experiment(
    model=mobilenet_model,
    freeze_function=(
        freeze_mobilenet_v3_backbone
    ),
    model_name="mobilenet_v3",
)

print(
    json.dumps(
        mobilenet_result,
        indent=2,
    )
)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to C:\Users\Dell/.cache\torch\hub\checkpoints\mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 29.8MB/s]


Postęp treningu:   0%|          | 0/10 [00:00<?, ?it/s]

Epoka 1/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 2/10:   0%|          | 0/445 [00:00<?, ?it/s]

...

Epoka 7/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 8/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 9/10:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 10/10:   0%|          | 0/445 [00:00<?, ?it/s]

{
  "model": "mobilenet_v3",
  "phase": "frozen",
  "accuracy": 0.8439490445859873,
  "macro_precision": 0.8500710674314795,
  "macro_recall": 0.8595159138870081,
  "macro_f1": 0.8402456222400445,
  "best_val_macro_f1": 0.8402456222400445,
  "training_time_seconds": 4606.7072348
}


## 5. Porównanie modeli przy zamrożonym backbone

In [9]:
frozen_results = pd.DataFrame(
    [
        resnet50_result,
        efficientnet_result,
        mobilenet_result,
    ]
)

frozen_results = (
    frozen_results
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    frozen_results
)

frozen_results.to_csv(
    METRICS_DIR
    / "architecture_frozen_validation_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

,model,phase,accuracy,macro_precision,macro_recall,macro_f1,best_val_macro_f1,training_time_seconds
0,resnet50,frozen,0.872213,0.866195,0.869549,0.858960,0.858960,16647.141073
1,mobilenet_v3,frozen,0.843949,0.850071,0.859516,0.840246,0.840246,4606.707235
2,efficientnet_b0,frozen,0.847134,0.803489,0.848840,0.819300,0.819300,7143.604567


## 6. Wybór architektury do fine-tuningu

Do drugiego etapu wybierana jest architektura osiągająca najwyższą wartość `validation macro F1` podczas treningu z zamrożonym backbone'em.

In [10]:
best_architecture = (
    frozen_results
    .iloc[0]["model"]
)

print(
    "Najlepsza architektura:",
    best_architecture,
)

Najlepsza architektura: resnet50


## 7. Fine-tuning najlepszego modelu

Najlepszy model zostaje częściowo odmrożony. Uczeniu podlegają końcowe warstwy reprezentacji oraz głowica klasyfikacyjna.

Wykorzystywany jest niższy learning rate, aby ograniczyć nadmierną modyfikację reprezentacji wyuczonych na ImageNet.

In [11]:
def run_fine_tuning(
    model,
    unfreeze_function,
    model_name,
):

    set_seed(
        RANDOM_SEED
    )

    model = unfreeze_function(
        model
    )

    model = model.to(
        DEVICE
    )

    criterion = (
        nn.CrossEntropyLoss()
    )

    optimizer = torch.optim.Adam(
        filter(
            lambda parameter:
            parameter.requires_grad,
            model.parameters(),
        ),
        lr=FINE_TUNE_LEARNING_RATE,
    )

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"{model_name}_finetuned_best.pt"
    )

    history = fit(
        model=model,
        train_loader=train_loader,
        val_loader=validation_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        epochs=FINE_TUNE_EPOCHS,
        checkpoint_path=checkpoint_path,
    )

    y_true, y_pred, inference_time = (
        predict(
            model,
            validation_loader,
            DEVICE,
        )
    )

    metrics = evaluate_predictions(
        y_true,
        y_pred,
    )

    result = {
        "model": model_name,
        "phase": "fine_tuning",
        "accuracy": metrics[
            "accuracy"
        ],
        "macro_precision": metrics[
            "macro_precision"
        ],
        "macro_recall": metrics[
            "macro_recall"
        ],
        "macro_f1": metrics[
            "macro_f1"
        ],
        "best_val_macro_f1": history[
            "best_val_macro_f1"
        ],
        "training_time_seconds": history[
            "training_time_seconds"
        ],
    }

    return (
        model,
        history,
        result,
    )

In [12]:
fine_tune_models = {}
fine_tune_histories = {}
fine_tune_results = {}

In [13]:
if best_architecture == "resnet50":

    (
        selected_model,
        selected_history,
        selected_result,
    ) = run_fine_tuning(
        model=resnet50_model,
        unfreeze_function=(
            unfreeze_resnet50_last_block
        ),
        model_name="resnet50",
    )


elif best_architecture == "efficientnet_b0":

    (
        selected_model,
        selected_history,
        selected_result,
    ) = run_fine_tuning(
        model=efficientnet_model,
        unfreeze_function=(
            unfreeze_efficientnet_b0_last_block
        ),
        model_name="efficientnet_b0",
    )


elif best_architecture == "mobilenet_v3":

    (
        selected_model,
        selected_history,
        selected_result,
    ) = run_fine_tuning(
        model=mobilenet_model,
        unfreeze_function=(
            unfreeze_mobilenet_v3_last_block
        ),
        model_name="mobilenet_v3",
    )


else:

    raise ValueError(
        "Nieznana architektura."
    )


print(
    json.dumps(
        selected_result,
        indent=2,
    )
)

Postęp treningu:   0%|          | 0/5 [00:00<?, ?it/s]

Epoka 1/5:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 2/5:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 3/5:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 4/5:   0%|          | 0/445 [00:00<?, ?it/s]

Epoka 5/5:   0%|          | 0/445 [00:00<?, ?it/s]

{
  "model": "resnet50",
  "phase": "fine_tuning",
  "accuracy": 0.9251592356687898,
  "macro_precision": 0.9181821959295218,
  "macro_recall": 0.9269991769832504,
  "macro_f1": 0.9183033782409022,
  "best_val_macro_f1": 0.9183033782409022,
  "training_time_seconds": 9674.039597399998
}


## 8. Końcowa ocena najlepszej konfiguracji

Po zakończeniu fazy transfer learningu i fine-tuningu najlepsza konfiguracja jest oceniana na niezależnym zbiorze `test_final`.

Zbiór ten nie był wykorzystywany do wyboru strategii balansowania ani architektury.

In [14]:
test_true, test_pred, test_time = (
    predict(
        selected_model,
        test_loader,
        DEVICE,
    )
)

test_metrics = (
    evaluate_predictions(
        test_true,
        test_pred,
    )
)

final_results = {
    "architecture": best_architecture,
    "balancing_strategy": (
        "weighted_sampler"
    ),
    "accuracy": test_metrics[
        "accuracy"
    ],
    "macro_precision": test_metrics[
        "macro_precision"
    ],
    "macro_recall": test_metrics[
        "macro_recall"
    ],
    "macro_f1": test_metrics[
        "macro_f1"
    ],
    "test_samples": len(
        test_dataset
    ),
    "inference_time_seconds": (
        test_time
    ),
}

(
    METRICS_DIR
    / "architecture_final_test.json"
).write_text(
    json.dumps(
        final_results,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        final_results,
        indent=2,
    )
)

{
  "architecture": "resnet50",
  "balancing_strategy": "weighted_sampler",
  "accuracy": 0.9001861330851559,
  "macro_precision": 0.8872076792222215,
  "macro_recall": 0.8925747430398414,
  "macro_f1": 0.8829800818611528,
  "test_samples": 4298,
  "inference_time_seconds": 364.59381200000644
}


In [15]:
test_report = pd.DataFrame(
    test_metrics[
        "classification_report"
    ]
).T

test_report.to_csv(
    METRICS_DIR
    / "architecture_final_test_per_class.csv",
    encoding="utf-8",
)

display(
    test_report
)

,precision,recall,f1-score,support
0,0.639344,0.780000,0.702703,50.000000
1,1.000000,0.818182,0.900000,11.000000
2,0.833333,1.000000,0.909091,10.000000
3,0.857143,0.857143,0.857143,21.000000
4,0.833333,0.909091,0.869565,11.000000
...,...,...,...,...
90,1.000000,1.000000,1.000000,11.000000
91,1.000000,1.000000,1.000000,5.000000
accuracy,0.900186,0.900186,0.900186,0.900186
macro avg,0.887208,0.892575,0.882980,4298.000000
